# [LES - nut- Smagorinsky] PitzDaily

## Preamble

In [115]:
# Standard Library
import sys
import os
from pathlib import Path
import foamnordic as fno
import numpy as np
import matplotlib.pyplot as plt
import onsaemiro as osm

# Machine Learning
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### Directory & Path

In [ ]:
# FoamNordic Project Directory
PROJECT_DIR = Path("/scratch/<allocation-account>/<user>")
CASE_TYPE = "les"
CASE_NAME = "pitzDailySmag"
BASE_DIR = PROJECT_DIR / "Codes" / "FoamNordic"
MAIN_DIR = BASE_DIR / "foamnordic_tutorials" / "incompressible"
OF_SCRIPT_DIR = BASE_DIR / "openfoam_tutorials" / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [117]:
# DataGraph Configuration
FIG_X, FIG_Y = 3.5, 2.55
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette("OKABE_ITO")
osm.set_style(figure_size=FIGURE_SIZE)

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>"
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

In [119]:
# FoamNordic configuration
SEED = 42
key = fno.Random.key(seed=SEED, scope="global")
key_jax = jax.random.PRNGKey(SEED)

## Example - Closure Modelling

### Synthetic Data Generation

In [120]:
# Smagorinsky model coefficients
C_K = 0.0265463553
C_E = 1.048

# Training configuration
N_TRAIN = 30000
N_NEIGHBORS = 5
N_ESTIMATORS = 100
N_SAMPLES = int(1e5)
BATCH_SIZE = 128
N_EPOCHS = 100
LEARNING_RATE = 1.0e-3

In [121]:
# Smagorinsky model for LES
def smagorinsky_nut(velocity_grad, delta):
    symm_grad = 0.5 * (velocity_grad + np.swapaxes(velocity_grad, -1, -2))

    trace_grad = np.trace(symm_grad, axis1=-2, axis2=-1)

    identity = np.eye(3, dtype=velocity_grad.dtype)

    dev_symm_grad = (
        symm_grad
        - (1.0 / 3.0)
        * trace_grad[..., None, None]
        * identity
    )

    dev_contraction = np.sum(
        dev_symm_grad * symm_grad,
        axis=(-2, -1),
    )

    coeff_a = C_E / delta
    coeff_b = (2.0 / 3.0) * trace_grad
    coeff_c = (2.0 * C_K * delta * dev_contraction)

    discriminant = (coeff_b**2 + 4.0 * coeff_a * coeff_c)

    sqrt_k = (
        -coeff_b
        + np.sqrt(np.maximum(discriminant, 0.0))
    ) / (
        2.0 * coeff_a
    )

    sqrt_k = np.maximum(sqrt_k, 0.0)

    nut = C_K * delta * sqrt_k

    return nut

In [122]:
# Load vanilla OpenFOAM results
post = fno.Postprocess.Case(OF_SCRIPT_DIR)
times = tuple(time for time in post.times if time > 0.0)

TRAIN_TIMES = times[:-1]
TEST_TIME = times[-1]

In [123]:
# Load cell volume and calculate LES filter width
cell_volume = post.field("V", physical_time=TEST_TIME).reshape(-1)
delta = np.cbrt(cell_volume)

# Load training trajectories
train_grad = []

for physical_time in TRAIN_TIMES:
    velocity_grad = post.field("gradU", physical_time=physical_time).reshape(-1, 3, 3)
    train_grad.append(velocity_grad)

train_grad = np.concatenate(train_grad)
train_delta = np.tile(delta, len(TRAIN_TIMES))

# Load the held-out final trajectory
test_grad = post.field("gradU", physical_time=TEST_TIME).reshape(-1, 3, 3)
test_delta = delta.copy()

print(f"Training times: {TRAIN_TIMES}")
print(f"Test time: {TEST_TIME}")
print(f"Training cells: {len(train_grad)}")
print(f"Test cells: {len(test_grad)}")

Training times: (0.0025, 0.005, 0.0075, 0.01, 0.0125, 0.015, 0.0175)
Test time: 0.02
Training cells: 85575
Test cells: 12225


In [124]:
# Calculate the mathematical Smagorinsky targets
y_train_full = smagorinsky_nut(velocity_grad=train_grad, delta=train_delta)
y_test = smagorinsky_nut(velocity_grad=test_grad, delta=test_delta)

# Pack grad(U) and delta into the FoamNordic model contract
X_train_full = np.column_stack([train_grad.reshape(len(train_grad), -1), train_delta])
X_test = np.column_stack([test_grad.reshape(len(test_grad), -1), test_delta])

# Select a stratified training subset
if len(X_train_full) > N_TRAIN:
    nut_bins = np.unique(np.quantile(y_train_full, np.linspace(0.0, 1.0, 11)))
    labels = np.digitize(y_train_full, nut_bins[1:-1])
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        train_size=N_TRAIN,
        test_size=min(5000, len(X_train_full) - N_TRAIN),
        random_state=SEED,
        stratify=labels,
    )
else:
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=SEED,
    )
    
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

Training samples: 30000
Validation samples: 5000
Test samples: 12225


In [125]:
# Feature Scaling
scaler_X = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train[:, None])

inactive_features = scaler_X.var_ < 1.0e-20

scaler_X.mean_[inactive_features] = 0.0
scaler_X.scale_[inactive_features] = 1.0
scaler_X.var_[inactive_features] = 1.0

X_train_scaled = scaler_X.transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.transform(y_train[:, None])[:, 0]
y_val_scaled = scaler_y.transform(y_val[:, None])[:, 0]

### ML Modelling

In [126]:
# Linear layer definition
class Linear(eqx.Module):
    weight: jax.Array
    bias: jax.Array

    def __init__(self, key, in_features, out_features, init="glorot"):
        if isinstance(init, str):
            init_map = {
                "glorot": jax.nn.initializers.glorot_normal(),
                "he": jax.nn.initializers.he_normal(),
                "lecun": jax.nn.initializers.lecun_normal(),
            }
            init = init_map.get(
                init.lower(),
                jax.nn.initializers.glorot_normal(),
            )

        self.weight = init(key, (out_features, in_features))
        self.bias = jnp.zeros(out_features, dtype=self.weight.dtype)

    def __call__(self, x):
        return self.weight @ x + self.bias

In [127]:
# Multi-layer Perceptron (MLP) definition
class MLP(eqx.Module):
    layers: list
    activation: callable = eqx.field(static=True)

    def __init__(
        self,
        key,
        input_dim,
        hidden_dims,
        output_dim,
        activation=jax.nn.relu,
        init="glorot",
    ):
        self.activation = activation

        dims = [input_dim, *hidden_dims, output_dim]
        keys = jax.random.split(key, len(dims) - 1)

        self.layers = [
            Linear(
                key=k,
                in_features=in_dim,
                out_features=out_dim,
                init=init,
            )
            for k, in_dim, out_dim in zip(keys, dims[:-1], dims[1:])
        ]

    def __call__(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))

        return self.layers[-1](x)

    def predict(self, X):
        return jax.vmap(self.__call__)(X)

In [128]:
# Trainer for the MLP model
class MLPTrainer(eqx.Module):
    model: eqx.Module
    optimizer: optax.GradientTransformation = eqx.field(static=True)
    opt_state: optax.OptState

    def __init__(self, model, learning_rate):
        self.model = model
        self.optimizer = optax.adam(learning_rate)
        self.opt_state = self.optimizer.init(
            eqx.filter(self.model, eqx.is_inexact_array)
        )

    def _predict_batch(self, X):
        return self.model.predict(X)[:, 0]

    def _loss_fn(self, model, X, y):
        y_pred = jax.vmap(model)(X)[:, 0]
        return jnp.mean((y_pred - y) ** 2)
    
    def loss(self, X, y):
        return self._loss_fn(self.model, X, y)

    @eqx.filter_jit
    def step(self, X, y):
        loss_value, grads = eqx.filter_value_and_grad(self._loss_fn)(
            self.model, X, y
        )

        updates, opt_state = self.optimizer.update(grads, self.opt_state, self.model)
        model = eqx.apply_updates(self.model, updates)

        trainer = eqx.tree_at(
            lambda t: (t.model, t.opt_state),
            self,
            (model, opt_state),
        )

        return trainer, loss_value

### Model Initialisation and Training

In [129]:
# Initialise model
input_dim = X_train_scaled.shape[1]
output_dim = 1
hidden_dims = [64, 64]
key_model, key_train = jax.random.split(key_jax)

smagorinsky_model = MLP(
    key=key_model,
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    output_dim=output_dim,
    activation=jax.nn.relu,
    init="glorot",
)

trainer = MLPTrainer(
    model=smagorinsky_model,
    learning_rate=LEARNING_RATE,
)

In [130]:
# Convert arrays to JAX format
X_train_jax = jnp.asarray(X_train_scaled)
y_train_jax = jnp.asarray(y_train_scaled)
X_val_jax = jnp.asarray(X_val_scaled)
y_val_jax = jnp.asarray(y_val_scaled)
X_test_jax = jnp.asarray(X_test_scaled)

In [131]:
# Hyperparameters
n_samples = X_train_jax.shape[0]
n_batches = n_samples // BATCH_SIZE

# Training monitoring
table = osm.TableMaker(
    title="DNN Training",
    columns=["Epoch", "Training MSE", "Validation MSE"],
    mode="dynamic",
)

# Training loop
for epoch in range(N_EPOCHS):
    key_train, subkey = jax.random.split(key_train)
    perm = jax.random.permutation(subkey, n_samples)
    
    X_shuffled = X_train_jax[perm]
    y_shuffled = y_train_jax[perm]
    
    training_mse = 0.0
    
    for i in range(n_batches):
        start_idx = i * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE
        
        X_batch = X_shuffled[start_idx:end_idx]
        y_batch = y_shuffled[start_idx:end_idx]
        
        trainer, batch_loss = trainer.step(X_batch, y_batch)
        training_mse += float(batch_loss)
        
    training_mse /= n_batches
    validation_mse = float(trainer.loss(X_val_jax, y_val_jax))
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        table.add_row([
            epoch + 1,
            f"{training_mse:.6e}",
            f"{validation_mse:.6e}",
        ])
        
# Update the model after training
smagorinsky_regressor = trainer.model

Epoch,Training MSE,Validation MSE
1,1.865502e-01,5.260754e-02
20,2.608237e-03,1.673581e-03
40,1.763760e-03,8.976870e-04
60,1.717598e-03,3.397160e-03
80,4.957061e-04,2.656514e-04
100,6.563940e-04,3.672018e-04


In [132]:
# Predict on the test set and inverse transform to physical scale
y_pred_scaled = np.asarray(smagorinsky_regressor.predict(X_test_jax)[:, 0])
y_pred = scaler_y.inverse_transform(y_pred_scaled[:, None])[:, 0]
y_pred = np.maximum(y_pred, 0.0)

# Compute metrics
metric_r2 = r2_score(y_test, y_pred)
metric_mse = mean_squared_error(y_test, y_pred)
metric_mae = mean_absolute_error(y_test, y_pred)
metric_relative_error = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1e-12))) * 100
metric_99_percentile_error = np.percentile(np.abs((y_test - y_pred) / np.maximum(y_test, 1e-12)), 99) * 100

table = osm.TableMaker(
    columns=["R2", "MSE", "MAE", "Relative Error (%)", "99th Percentile Error (%)"],
    title="Smagorinsky Model Performance Metrics",
    mode="static",
)

table.add_row([
    f"{metric_r2:.6f}",
    f"{metric_mse:.6e}",
    f"{metric_mae:.6e}",
    f"{metric_relative_error:.6f}",
    f"{metric_99_percentile_error:.6f}",
])

table.display()

R2,MSE,MAE,Relative Error (%),99th Percentile Error (%)
0.995397,1.621576e-13,8.044785e-08,21.342632,153.382833


In [133]:
# Model Export
# Define the FoamNordic model path
MODEL_PATH = MODEL_DIR / "smagorinsky_eqx.fnom"

# Export the trained model in FoamNordic format
fno.Export.equinox(
    smagorinsky_regressor,
    path=MODEL_PATH,
    inputs={
        "velocity_grad": fno.Tensor.tensor(),
        "delta": fno.Tensor.scalar(),
    },
    outputs={
        "nut": fno.Tensor.scalar(),
    },
    x_scaler=scaler_X,
    y_scaler=scaler_y,
    batched=False,
    verbose=True,
);

Property,Value
Manifest,smagorinsky_eqx.fnom
Payload,smagorinsky_eqx.eqx
Format,Equinox + FNOM v1
Dtype,float64
Inputs,"velocity_grad[9], delta[1]"
Outputs,nut[1]
Input scaler,standard
Output scaler,standard
Compression,none (path-backed startup)
Payload size,33348 B


### Smagorinsky Closure

In [134]:
# Define the Smagorinsky closure
smagorinsky_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.model(MODEL_PATH),
    inputs={
        "velocity_grad": fno.Field.grad("U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "nut": fno.Field("nut"),
    },
)

### Case Definition

In [135]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [136]:
# Connect the OpenFOAM case and SLURM scheduler (launch a longship instance)
longship = fno.Longship(case=case, closures=(smagorinsky_closure,))

# Set sail for the OpenFOAM case (submit the job to the HPC cluster)
run = longship.launch(start_timeout=900)

[FoamNordic] Preparing mesh with blockMesh: pitzDaily
[FoamNordic] Mesh is ready: pitzDaily
[FoamNordic] Sailing in background: pitzDaily


In [137]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [140]:
# Summary of the job result
result.summary(style="compact");

Job ID,Name,Status,Partition,Node,Elapsed
-,pitzDaily,succeeded,local,rc5183,00:00:59


### Postprocessing

In [141]:
# Postprocessing
post = result.postprocess

velocity = post.field("U", time_idx=-1)
pressure = post.field("p", time_idx=-1)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    ["U", "p", "nut"],
    time_idx=-1,
    verbose=True,
)

U shape: (12225, 3)
p shape: (12225,)


Field,Min,Max,Mean,Std,RMS
U,1.574077e-02,1.384266e+01,6.369703e+00,2.648083e+00,6.898222e+00
p,-3.556820e+01,1.143490e+02,6.504298e+01,2.181318e+01,6.860324e+01
nut,-8.077290e-07,1.043300e-04,2.488986e-06,5.571311e-06,6.102012e-06
